# 01-parsbert-finetune

Account A. ParsBERT, the project primary. Persian WordPiece. Persian digits stay as they are.

Target classes: `recommended`, `not_recommended`, `no_idea`. The only model input is `body`. The only label is `recommendation_status`. The notebook reads those two columns from the preprocessed CSV and no other field.

This file is one of four. Run each file on its own Colab account at the same time:

- `01-parsbert-finetune.ipynb`
- `02-xlmr-finetune.ipynb`
- `03-tookabert-finetune.ipynb`
- `04-shiraz-finetune.ipynb`

Upload one CSV to Drive: `digikala-comments-preprocessed.csv`. Do not upload the raw comments file, the cleaned CSV, or the tokenized parquet. Those token ids belong to ParsBERT and the other three models cannot read them.

Expected Drive path after you add a shortcut into My Drive:

`MyDrive/digikala-experiments/data/digikala-comments-preprocessed.csv`

Each notebook writes its own folder under `MyDrive/digikala-experiments/runs/` and `MyDrive/digikala-experiments/best/`. Stage 1 trains only the classifier. Stage 2 unfreezes the last encoder block. Validation macro F1 picks the checkpoint. Test accuracy, precision, recall, and F1 are written once, into that model's `best` folder.

Do not use `HooshvareLab/bert-fa-base-uncased-sentiment-digikala`.


In [ ]:
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
RUN_STAGE = "both"  # "head", "last_block", or "both"
MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
HEAD_LR = 1e-3
BLOCK_LR = 2e-5
HEAD_LR_WHEN_UNFROZEN = 1e-4
SEED = 2026

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
DATA_CSV = f"{DRIVE_DIR}/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
SPLITS = ["train", "validation", "test"]
SPLIT_SHARE = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}


## Drive and the preprocessed CSV

Read only `body` and `recommendation_status` from `digikala-comments-preprocessed.csv`. Mount Drive, then confirm the file is there before any training starts.


In [ ]:
from pathlib import Path
import json

import numpy as np
import polars as pl
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

drive.mount("/content/drive")
data_path = Path(DATA_CSV)
if not data_path.is_file():
    raise FileNotFoundError(
        f"Put the preprocessed CSV at {DATA_CSV} before training."
    )
drive_root = Path(DRIVE_DIR)
(drive_root / "runs").mkdir(parents=True, exist_ok=True)
(drive_root / "best").mkdir(parents=True, exist_ok=True)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("data:", data_path, "bytes", data_path.stat().st_size)


## Stratified split

Each class is shuffled with seed 2026 and cut 70/15/15 into train, validation, and test. The cut uses `recommendation_status` only.


In [ ]:
def fold_digits_for_tokenizer(body: pl.Expr, model_name: str) -> pl.Expr:
    """XLM-R was trained on ASCII digits. ParsBERT and TookaBERT keep Persian digits."""
    if "xlm-roberta" not in model_name.lower():
        return body
    folded = body
    for persian_digit, ascii_digit in zip("۰۱۲۳۴۵۶۷۸۹", "0123456789"):
        folded = folded.str.replace_all(persian_digit, ascii_digit, literal=True)
    return folded


reviews = pl.read_csv(
    DATA_CSV,
    columns=["body", "recommendation_status"],
    schema_overrides={
        "body": pl.String,
        "recommendation_status": pl.String,
    },
)
reviews = reviews.with_columns(
    fold_digits_for_tokenizer(pl.col("body"), MODEL_NAME).alias("body")
)
labeled = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
)

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(
        shuffled.with_columns(
            pl.Series(
                "split",
                (["train"] * n_train)
                + (["validation"] * n_validation)
                + (["test"] * n_test),
            )
        )
    )
labeled = pl.concat(parts)
print(
    labeled.group_by("split", "recommendation_status")
    .len()
    .sort("split", "recommendation_status")
)


## Classifier head, then the last encoder block

Stage `head` trains only the new classification layer. Stage `last_block` starts from those weights, unfreezes the top encoder block, and uses a smaller learning rate on that block than on the classifier.


In [ ]:
class ReviewSet(Dataset):
    def __init__(self, frame: pl.DataFrame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average="macro", zero_division=0
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average=None, zero_division=0
    )
    report = {
        "accuracy": float(accuracy_score(actual, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    for index, name in enumerate(LABELS):
        report[f"{name}_precision"] = float(per_precision[index])
        report[f"{name}_recall"] = float(per_recall[index])
        report[f"{name}_f1"] = float(per_f1[index])
        report[f"{name}_support"] = int(per_support[index])
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted = np.argmax(logits, axis=-1)
    scores = metric_bundle(labels, predicted)
    return {key: scores[key] for key in ("accuracy", "precision", "recall", "f1")}


def encoder_prefix(model) -> str:
    for prefix in ("bert", "roberta", "distilbert", "mobilebert"):
        if hasattr(model, prefix):
            return prefix
    raise AttributeError("This checkpoint has no bert/roberta/distilbert/mobilebert encoder.")


def freeze_encoder(model) -> None:
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(prefix + ".")


def unfreeze_last_block(model) -> None:
    freeze_encoder(model)
    prefix = encoder_prefix(model)
    encoder = getattr(model, prefix).encoder
    layers = getattr(encoder, "layer", None)
    if layers is None:
        layers = encoder.layers
    for param in layers[-1].parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" in name or name.startswith("score"):
            param.requires_grad = True


def parameter_groups(model, block_lr: float, head_lr: float):
    head, block = [], []
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(prefix + "."):
            block.append(param)
        else:
            head.append(param)
    return [
        {"params": block, "lr": block_lr},
        {"params": head, "lr": head_lr},
    ]


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def slug(model_name: str) -> str:
    return model_name.replace("/", "--")


def write_run(stage: str, metrics: dict, model, tokenizer) -> Path:
    run_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(run_dir)
    tokenizer.save_pretrained(run_dir)
    payload = {"model_name": MODEL_NAME, "stage": stage, "max_length": MAX_LENGTH, **metrics}
    (run_dir / "metrics.json").write_text(json.dumps(payload, indent=2))
    log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}.jsonl"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as handle:
        handle.write(json.dumps(payload) + "\n")
    print("saved", run_dir)
    print(json.dumps({key: payload[key] for key in ("accuracy", "precision", "recall", "f1")}, indent=2))
    return run_dir


def promote_if_best(run_dir: Path, metrics: dict) -> None:
    best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
    best_dir.mkdir(parents=True, exist_ok=True)
    best_metrics_path = best_dir / "metrics.json"
    previous = -1.0
    if best_metrics_path.is_file():
        previous = json.loads(best_metrics_path.read_text()).get("f1", -1.0)
    if metrics["f1"] <= previous:
        print("validation macro F1", round(metrics["f1"], 4), "does not beat", round(previous, 4))
        return
    for source in run_dir.iterdir():
        target = best_dir / source.name
        target.write_bytes(source.read_bytes())
    print("best model updated", best_dir, "macro F1", round(metrics["f1"], 4))


## Train

`RUN_STAGE = "both"` trains the head, reloads that checkpoint, then trains the last block. Change `MODEL_NAME` on the other Colab account. Do not run two stages of the same checkpoint on two accounts at once, or they will overwrite the same Drive folder.


In [ ]:
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_frame = labeled.filter(pl.col("split") == "train")
valid_frame = labeled.filter(pl.col("split") == "validation")
test_frame = labeled.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)

class_weights = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.array([0, 1, 2]),
        y=train_frame.get_column("label_id").to_numpy(),
    ),
    dtype=torch.float,
)
train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)


def make_args(stage: str, learning_rate: float) -> TrainingArguments:
    return TrainingArguments(
        output_dir=f"/content/checkpoints/{slug(MODEL_NAME)}-{stage}",
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )


def score_frame(model, frame: pl.DataFrame) -> dict:
    holder = ReviewSet(frame, tokenizer, MAX_LENGTH)
    args = TrainingArguments(
        output_dir="/tmp/score",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args, data_collator=collator, tokenizer=tokenizer)
    prediction = trainer.predict(holder)
    return metric_bundle(prediction.label_ids, np.argmax(prediction.predictions, axis=-1))


def train_stage(stage: str, model) -> None:
    if stage == "head":
        freeze_encoder(model)
        learning_rate = HEAD_LR
        groups = None
    elif stage == "last_block":
        unfreeze_last_block(model)
        learning_rate = BLOCK_LR
        groups = parameter_groups(model, BLOCK_LR, HEAD_LR_WHEN_UNFROZEN)
    else:
        raise ValueError(stage)
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(stage, "trainable parameters", trainable)
    trainer = WeightedTrainer(
        model=model,
        args=make_args(stage, learning_rate),
        train_dataset=train_set,
        eval_dataset=valid_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        tokenizer=tokenizer,
    )
    if groups is not None:
        def create_optimizer():
            trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
            return trainer.optimizer
        trainer.create_optimizer = create_optimizer
    trainer.train()
    metrics = score_frame(model, valid_frame)
    metrics["split"] = "validation"
    run_dir = write_run(stage, metrics, model, tokenizer)
    promote_if_best(run_dir, metrics)


stages = ["head", "last_block"] if RUN_STAGE == "both" else [RUN_STAGE]
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={index: name for index, name in enumerate(LABELS)},
    label2id=LABEL_CODE,
)
for stage in stages:
    if stage == "last_block":
        head_dir = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--head"
        if not head_dir.is_dir():
            raise FileNotFoundError(f"Train the head stage first. Missing {head_dir}")
        model = AutoModelForSequenceClassification.from_pretrained(head_dir)
        tokenizer = AutoTokenizer.from_pretrained(head_dir)
        train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
        valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
        collator = DataCollatorWithPadding(tokenizer)
    train_stage(stage, model)

best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
best_dir.mkdir(parents=True, exist_ok=True)
best_model = AutoModelForSequenceClassification.from_pretrained(best_dir)
tokenizer = AutoTokenizer.from_pretrained(best_dir)
test_metrics = score_frame(best_model, test_frame)
test_metrics["split"] = "test"
test_metrics["model_name"] = json.loads((best_dir / "metrics.json").read_text()).get("model_name")
(best_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("test", json.dumps({key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")}, indent=2))
